# Multi-Confirmation Trading Bot (FULL)

**Default lookback = DAYS**. Works with **Binance** and **Kraken** OHLCV **without API keys** (public endpoints via CCXT). **Forex** uses **OANDA** and requires an API key.

What you get:
- Broker choice in the Config cell: `binance` | `kraken` | `forex`.
- Lookback by **days** (default) or **bars**.
- EMA crossover + ADX + ATR filters + Higher Timeframe (HTF) confirmation.
- Optional **second Higher Timeframe (HTF2)** for 3-timeframe confluence (all three must agree for a trade).
- **Live trading mode** with real order execution via CCXT (Binance/Kraken).
- **Dynamic position sizing** — recalculates from live balance before every trade.
- **Email notifications** on every trade open with order details and current balance.
- Backtester and console Paper Trader inside this notebook.

> If needed, install dependencies in the next cell.

1) Install dependencies

In [11]:
# Uncomment and run once:
# !pip install -q pandas numpy ccxt oandapyV20 python-dotenv

2) Configuration

In [ ]:
import os
import logging
from dotenv import load_dotenv
load_dotenv(override=True)

# ── Mode ──────────────────────────────────────────────────────────────────────
# backtest → historical simulation, no keys needed
# paper    → live signals, simulated fills, no keys needed
# live     → real orders submitted to the broker via API
MODE = os.getenv("MODE").lower()

# ── Broker ────────────────────────────────────────────────────────────────────
# binance | bybit | kraken | forex
#
# For BTC/USDT perpetual futures (long + short) use:
#   BROKER = bybit   →  available in Canada, full futures support
#   BROKER = binance →  NOT available in Canada (blocked since 2023)
BROKER = os.getenv("BROKER", "binance").lower()

# ── Asset ─────────────────────────────────────────────────────────────────────
# Binance/Bybit futures : BTC/USDT:USDT  (USDT-margined perpetual)
# Binance/Bybit spot    : BTC/USDT
# Kraken                : BTC/USDT
# OANDA (forex)         : EUR_USD, GBP_USD
SYMBOL = os.getenv("SYMBOL", "BTC/USDT:USDT")

# ── Timeframes ────────────────────────────────────────────────────────────────
# Options: "1m" "5m" "15m" "30m" "1h" "4h" "1d"
LTF = os.getenv("LTF")   # entry timeframe
HTF = os.getenv("HTF")   # first trend filter

# ── HTF2 (second trend filter — 3-TF confluence) ──────────────────────────────
# True  → LONG  only when HTF2 bull + HTF bull + LTF cross up
#         SHORT only when HTF2 bear + HTF bear + LTF cross down
# False → original 2-TF logic
# ↓ Edit directly — True = 3-TF confluence | False = 2-TF only
USE_HTF2 = os.getenv("USE_HTF2")

HTF2            = os.getenv("HTF2", "4h")
HTF2_BARS       = int(os.getenv("HTF2_BARS", "500"))
PAPER_HTF2_DAYS = int(os.getenv("PAPER_HTF2_DAYS", "120"))

# ── Lookback ──────────────────────────────────────────────────────────────────
DATA_PULL_MODE  = os.getenv("DATA_PULL_MODE", "days").lower()  # days | bars
LTF_BARS        = int(os.getenv("LTF_BARS", "1500"))
HTF_BARS        = int(os.getenv("HTF_BARS", "1500"))
BACKTEST_DAYS   = int(os.getenv("BACKTEST_DAYS", "30"))
PAPER_LTF_DAYS  = int(os.getenv("PAPER_LTF_DAYS", "30"))
PAPER_HTF_DAYS  = int(os.getenv("PAPER_HTF_DAYS", "60"))

# ── Strategy params ───────────────────────────────────────────────────────────
EMA_FAST      = int(os.getenv("EMA_FAST", "9"))
EMA_SLOW      = int(os.getenv("EMA_SLOW", "21"))
ADX_PERIOD    = int(os.getenv("ADX_PERIOD", "14"))
ADX_THRESHOLD = float(os.getenv("ADX_THRESHOLD","35"))  
ATR_PERIOD    = int(os.getenv("ATR_PERIOD", "14"))
ATR_SL_MULT   = float(os.getenv("ATR_SL_MULT", "1.5"))
ATR_TP_MULT   = float(os.getenv("ATR_TP_MULT", "10"))

# ── Risk & capital ────────────────────────────────────────────────────────────
# RISK_PCT        : % of current balance to risk per trade (applied dynamically in live mode)
# INITIAL_CAPITAL : starting balance for backtest/paper simulation only
#                   in live mode the bot fetches the real balance from the exchange
RISK_PCT        = float(os.getenv("RISK_PCT", "3.0"))
INITIAL_CAPITAL = float(os.getenv("INITIAL_CAPITAL", "100"))

# ── Live mode: dynamic position sizing ───────────────────────────────────────
# LIVE_QUOTE_ASSET : the asset your balance is held in (USDT, USDC, USD, etc.)
# LIVE_LEVERAGE    : leverage multiplier (1.0 = no leverage / spot)
#                    Note: only relevant if your broker supports margin/futures
LIVE_QUOTE_ASSET = os.getenv("LIVE_QUOTE_ASSET", "USDT")
LIVE_LEVERAGE    = float(os.getenv("LIVE_LEVERAGE", "1.0"))

# ── Poll interval ─────────────────────────────────────────────────────────────
PAPER_POLL_INTERVAL = int(os.getenv("PAPER_POLL_INTERVAL", "60"))  # seconds
PAPER_MAX_TRADES    = int(os.getenv("PAPER_MAX_TRADES", "0"))       # 0 = unlimited

# ── Long-only mode (spot trading) ────────────────────────────────────────────
# Set LONG_ONLY = True  when trading on a spot exchange (e.g. Kraken spot)
# that does not support short selling.
# SHORT signals will be completely ignored in backtest, paper, and live modes.
# Set LONG_ONLY = False to trade both directions (requires futures/margin).
# ↓ Edit this line directly — True = spot long-only | False = long + short
LONG_ONLY = os.getenv("LONG_ONLY")

LONG_ONLY_LABEL = "ON (spot — longs only)" if LONG_ONLY else "OFF (long + short)"

# ── Email notifications ───────────────────────────────────────────────────────
# Sent on every live trade open containing order details + current balance.
# Uses Gmail SMTP by default. For Gmail you must use an App Password, not your
# regular password: myaccount.google.com/apppasswords
#
# Set ENABLE_EMAIL = False to disable without removing the config.
ENABLE_EMAIL     = os.getenv("ENABLE_EMAIL", "true").lower() == "true"
EMAIL_SENDER     = os.getenv("EMAIL_SENDER", "")       # your gmail address
EMAIL_PASSWORD   = os.getenv("EMAIL_PASSWORD", "")     # Gmail App Password
EMAIL_RECIPIENT  = os.getenv("EMAIL_RECIPIENT", "")    # where to send alerts
EMAIL_SMTP_HOST  = os.getenv("EMAIL_SMTP_HOST", "smtp.gmail.com")
EMAIL_SMTP_PORT  = int(os.getenv("EMAIL_SMTP_PORT", "587"))

# ── Broker API keys ───────────────────────────────────────────────────────────
# Not required for data fetching (public endpoints).
# REQUIRED for live mode order execution.
BINANCE_API_KEY    = os.getenv("BINANCE_API_KEY", "")
BINANCE_API_SECRET = os.getenv("BINANCE_API_SECRET", "")
BYBIT_API_KEY      = os.getenv("BYBIT_API_KEY", "")      # recommended for Canada
BYBIT_API_SECRET   = os.getenv("BYBIT_API_SECRET", "")
KRAKEN_API_KEY     = os.getenv("KRAKEN_API_KEY", "")
KRAKEN_API_SECRET  = os.getenv("KRAKEN_API_SECRET", "")
OANDA_API_KEY      = os.getenv("OANDA_API_KEY", "")
OANDA_ACCOUNT_ID   = os.getenv("OANDA_ACCOUNT_ID", "")
OANDA_ENVIRONMENT  = os.getenv("OANDA_ENVIRONMENT", "practice")  # practice | live

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger("bot")

# ── Startup summary ───────────────────────────────────────────────────────────
htf2_label   = f" / {HTF2} (ON)" if USE_HTF2 else " (HTF2 OFF)"
email_label  = f"ON → {EMAIL_RECIPIENT}" if ENABLE_EMAIL and EMAIL_RECIPIENT else "OFF"
print(
    "Config:\n"
    f"  Mode          : {MODE}\n"
    f"  Broker        : {BROKER}  Symbol: {SYMBOL}\n"
    f"  LTF/HTF/HTF2  : {LTF} / {HTF}{htf2_label}\n"
    f"  Lookback mode : {DATA_PULL_MODE}  LTF_BARS={LTF_BARS}  HTF_BARS={HTF_BARS}  HTF2_BARS={HTF2_BARS}\n"
    f"                  BACKTEST_DAYS={BACKTEST_DAYS}  PAPER_L/H/H2={PAPER_LTF_DAYS}/{PAPER_HTF_DAYS}/{PAPER_HTF2_DAYS}\n"
    f"  Capital       : ${INITIAL_CAPITAL:,.2f} (sim)  Risk/Trade: {RISK_PCT}%  Leverage: {LIVE_LEVERAGE}x\n"
    f"  Email alerts  : {email_label}\n"
    f"  Long-only mode : {LONG_ONLY_LABEL}\n"
    f"  Take Profit: {ATR_TP_MULT}\n"
    f"  Take Profit: {ATR_SL_MULT}\n"
)

if MODE == 'live':
    if BROKER == 'binance' and (not BINANCE_API_KEY or not BINANCE_API_SECRET):
        raise EnvironmentError("Live mode requires BINANCE_API_KEY and BINANCE_API_SECRET in your .env")
    if BROKER == 'kraken' and (not KRAKEN_API_KEY or not KRAKEN_API_SECRET):
        raise EnvironmentError("Live mode requires KRAKEN_API_KEY and KRAKEN_API_SECRET in your .env")
    if ENABLE_EMAIL and (not EMAIL_SENDER or not EMAIL_PASSWORD or not EMAIL_RECIPIENT):
        log.warning("Email enabled but EMAIL_SENDER / EMAIL_PASSWORD / EMAIL_RECIPIENT not all set — emails will be skipped.")
    print("LIVE MODE — real orders will be placed on the exchange. Proceed carefully.\n")

15.0
Config:
  Mode          : backtest
  Broker        : binance  Symbol: BTC/USDT:USDT
  LTF/HTF/HTF2  : 1m / 30m (HTF2 OFF)
  Lookback mode : days  LTF_BARS=1500  HTF_BARS=1500  HTF2_BARS=500
                  BACKTEST_DAYS=30  PAPER_L/H/H2=30/60/120
  Capital       : $100.00 (sim)  Risk/Trade: 3.0%  Leverage: 15.0x
  Email alerts  : ON → tradingposition85@gmail.com
  Long-only mode : OFF (long + short)
  Take Profit: 1.7
  Take Profit: 1.5



3) Data Fetchers

In [13]:
import time
from datetime import datetime, timezone
import numpy as np
import pandas as pd

_TF_MIN = {"1m":1,"3m":3,"5m":5,"15m":15,"30m":30,"1h":60,"2h":120,"4h":240,"6h":360,"12h":720,"1d":1440}

# ── Data source ────────────────────────────────────────────────────────────────
# DATA_SOURCE controls where price data is pulled from for ALL modes
# (backtest, paper and live). This is independent of BROKER which only
# controls where live orders are sent.
# Binance public OHLCV: free, no API key needed, deep history, full pagination.
# Kraken is capped at 720 candles per call with no pagination — not recommended.
DATA_SOURCE = "binance"

def _ccxt_client(name: str, key: str = "", secret: str = ""):
    try:
        import ccxt
    except Exception as e:
        raise RuntimeError("ccxt is required. Install with: pip install ccxt") from e
    opts = {
        "enableRateLimit": True,
        "options": {"defaultType": "spot", "adjustForTimeDifference": True},
        "timeout": 20000,
    }
    if key:
        opts.update({"apiKey": key or None, "secret": secret or None})
    return getattr(ccxt, name)(opts)

def _days_to_bars(timeframe: str, days: int) -> int:
    mins = _TF_MIN.get(timeframe, 60)
    return max(500, int(days * 1440 / mins))

def _fetch_ccxt_ohlcv_paginated(ex, symbol: str, timeframe: str, candles: int) -> pd.DataFrame:
    """
    Fetch exactly `candles` bars by calculating the correct start timestamp
    and paging forward in time. This correctly handles large requests e.g.
    30 days of 1m candles = 43200 bars across 44 paginated calls.
    """
    per_call = 1000
    mins     = _TF_MIN.get(timeframe, 60)
    tf_ms    = mins * 60 * 1000
    now_ms   = int(datetime.now(timezone.utc).timestamp() * 1000)
    since    = now_ms - (candles * tf_ms)   # start this many bars in the past
    out      = []

    while len(out) < candles:
        batch = min(per_call, candles - len(out))
        ohlcv = ex.fetch_ohlcv(symbol, timeframe, since=since, limit=batch)
        if not ohlcv:
            break
        out.extend(ohlcv)
        since = ohlcv[-1][0] + tf_ms        # next page starts after last received candle
        rl = getattr(ex, "rateLimit", None)
        if rl:
            time.sleep(rl / 1000.0)
        if len(ohlcv) < batch:
            break                           # exchange returned less than asked — end of history

    df = pd.DataFrame(out, columns=["timestamp","open","high","low","close","volume"]).astype(
        {"open":float,"high":float,"low":float,"close":float,"volume":float}
    )
    if df.empty:
        return df
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df.drop_duplicates(subset="timestamp").set_index("timestamp").sort_index()
    return df

def fetch_binance(symbol: str, timeframe: str, *, bars: int=None, days: int=None) -> pd.DataFrame:
    ex = _ccxt_client("binance", BINANCE_API_KEY, BINANCE_API_SECRET)
    n  = bars if bars else _days_to_bars(timeframe, days or 30)
    return _fetch_ccxt_ohlcv_paginated(ex, symbol, timeframe, n)

def fetch_kraken(symbol: str, timeframe: str, *, bars: int=None, days: int=None) -> pd.DataFrame:
    ex = _ccxt_client("kraken", KRAKEN_API_KEY, KRAKEN_API_SECRET)
    # Kraken hard limit 720 candles, no pagination. Use DATA_SOURCE=binance for deeper history.
    n  = min(bars if bars else _days_to_bars(timeframe, days or 30), 720)
    return _fetch_ccxt_ohlcv_paginated(ex, symbol, timeframe, n)

def fetch_forex(symbol: str, timeframe: str, *, bars: int=None, days: int=None) -> pd.DataFrame:
    try:
        import oandapyV20
        import oandapyV20.endpoints.instruments as instruments
    except Exception as e:
        raise RuntimeError("oandapyV20 is required for forex. Install with: pip install oandapyV20") from e
    tf_map = {"1m":"M1","5m":"M5","15m":"M15","30m":"M30","1h":"H1","4h":"H4","1d":"D"}
    if not OANDA_API_KEY:
        raise RuntimeError("OANDA_API_KEY is required for BROKER='forex'.")
    client = oandapyV20.API(access_token=OANDA_API_KEY, environment=OANDA_ENVIRONMENT)
    count  = bars or max(500, (days or 30) * 48)
    req    = instruments.InstrumentsCandles(
        instrument=symbol,
        params={"granularity": tf_map.get(timeframe, "H1"), "count": int(count)}
    )
    client.request(req)
    rows = []
    for c in req.response.get("candles", []):
        if not c.get("complete", False):
            continue
        mid = c.get("mid") or {}
        rows.append({
            "timestamp": pd.to_datetime(c["time"]),
            "open":  float(mid.get("o", 0)),
            "high":  float(mid.get("h", 0)),
            "low":   float(mid.get("l", 0)),
            "close": float(mid.get("c", 0)),
            "volume":int(c.get("volume", 0))
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.set_index("timestamp").sort_index()

def fetch_demo_data(symbol: str, timeframe: str, *, days: int=365) -> pd.DataFrame:
    mins  = _TF_MIN.get(timeframe, 60)
    n     = max(300, int(days * 1440 / mins))
    rng   = pd.date_range(end=datetime.now(), periods=n, freq=f"{mins}min")
    np.random.seed(42)
    lr    = np.random.normal(0.0001, 0.002, n)
    block = max(1, n // 6)
    for i in range(0, n, block):
        lr[i:i+block] += np.random.choice([-1,1]) * 0.0003
    prices = 100 * np.exp(np.cumsum(lr))
    high   = prices * (1 + np.abs(np.random.normal(0, 0.003, n)))
    low    = prices * (1 - np.abs(np.random.normal(0, 0.003, n)))
    open_  = np.roll(prices, 1); open_[0] = prices[0]
    vol    = np.random.randint(1000, 50000, n).astype(float)
    return pd.DataFrame({"open":open_,"high":high,"low":low,"close":prices,"volume":vol}, index=rng)

def get_data(symbol: str, timeframe: str, *, bars: int=None, days: int=None) -> pd.DataFrame:
    """
    Pull OHLCV data from DATA_SOURCE — independent of BROKER.
    Falls back to DEMO synthetic data only if the real fetch fails.
    Prints source, candle count, span, and which broker will execute orders.
    """
    src_name = DATA_SOURCE.lower()
    source   = src_name.upper()
    try:
        if src_name == "binance":
            df = fetch_binance(symbol, timeframe, bars=bars, days=days)
        elif src_name == "kraken":
            df = fetch_kraken(symbol, timeframe, bars=bars, days=days)
        elif src_name == "forex":
            df = fetch_forex(symbol, timeframe, bars=bars, days=days)
        else:
            log.warning(f"Unknown DATA_SOURCE '{src_name}' — using DEMO synthetic data.")
            df     = fetch_demo_data(symbol, timeframe, days=days or 30)
            source = "DEMO"
    except Exception as e:
        log.error(f"Data fetch failed ({src_name}): {e} — using DEMO synthetic.")
        df     = fetch_demo_data(symbol, timeframe, days=days or 30)
        source = "DEMO"

    candles = len(df)
    if candles:
        span_days = (df.index[-1] - df.index[0]).days
        print(
            f"Data pull {symbol:<12} {timeframe:<4} "
            f"src: {source:<8} "
            f"candles: {candles:>6}  span: {span_days}d "
            f"({df.index[0].strftime('%Y-%m-%d')} -> {df.index[-1].strftime('%Y-%m-%d')})  "
            f"[execute on: {BROKER.upper()}]"
        )
    else:
        print(f"Data pull returned 0 candles. src: {source}  [execute on: {BROKER.upper()}]")
    return df


4) Indicators

In [14]:
import numpy as np
import pandas as pd

def ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=period, adjust=False).mean()

def calc_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    hl = df['high'] - df['low']
    hc = (df['high'] - df['close'].shift()).abs()
    lc = (df['low'] - df['close'].shift()).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()

def calc_adx(df: pd.DataFrame, period: int = 14) -> pd.Series:
    up = df['high'].diff()
    down = -df['low'].diff()
    pdm = np.where((up > down) & (up > 0), up, 0.0)
    mdm = np.where((down > up) & (down > 0), down, 0.0)
    tr = calc_atr(df, period)
    pdi = 100 * pd.Series(pdm, index=df.index).ewm(span=period, adjust=False).mean() / tr
    mdi = 100 * pd.Series(mdm, index=df.index).ewm(span=period, adjust=False).mean() / tr
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    return dx.ewm(span=period, adjust=False).mean()

def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['ema_fast'] = ema(df['close'], EMA_FAST)
    df['ema_slow'] = ema(df['close'], EMA_SLOW)
    df['atr'] = calc_atr(df, ATR_PERIOD)
    df['atr_ma'] = df['atr'].rolling(ATR_PERIOD * 2).mean()
    df['adx'] = calc_adx(df, ADX_PERIOD)
    return df.dropna()

5) Signal Generation

In [15]:
import numpy as np
import pandas as pd

def generate_signals(ltf_df: pd.DataFrame, htf_df: pd.DataFrame,
                     htf2_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    LONG  : HTF2 bull (if ON) + HTF bull + LTF EMA cross up   + ADX > threshold + ATR expanding
    SHORT : HTF2 bear (if ON) + HTF bear + LTF EMA cross down + ADX > threshold + ATR expanding
    """
    df = ltf_df.copy()
    htf_trend = (htf_df['ema_fast'] > htf_df['ema_slow']).astype(int).reindex(df.index, method='ffill')

    if USE_HTF2 and htf2_df is not None:
        htf2_trend = (htf2_df['ema_fast'] > htf2_df['ema_slow']).astype(int).reindex(df.index, method='ffill')
    else:
        htf2_trend = None

    cross_up   = (df['ema_fast'] > df['ema_slow']) & (df['ema_fast'].shift() <= df['ema_slow'].shift())
    cross_down = (df['ema_fast'] < df['ema_slow']) & (df['ema_fast'].shift() >= df['ema_slow'].shift())
    trend_ok   = df['adx'] > ADX_THRESHOLD
    vol_ok     = df['atr'] > df['atr_ma']

    df['signal'] = 0
    if USE_HTF2 and htf2_trend is not None:
        df.loc[cross_up   & trend_ok & vol_ok & (htf_trend == 1) & (htf2_trend == 1), 'signal'] =  1
        if not LONG_ONLY:
            df.loc[cross_down & trend_ok & vol_ok & (htf_trend == 0) & (htf2_trend == 0), 'signal'] = -1
    else:
        df.loc[cross_up   & trend_ok & vol_ok & (htf_trend == 1), 'signal'] =  1
        if not LONG_ONLY:
            df.loc[cross_down & trend_ok & vol_ok & (htf_trend == 0), 'signal'] = -1

    sign = df['signal']
    df['sl'] = np.where(sign ==  1, df['close'] - ATR_SL_MULT * df['atr'],
                np.where(sign == -1, df['close'] + ATR_SL_MULT * df['atr'], np.nan))
    df['tp'] = np.where(sign ==  1, df['close'] + ATR_TP_MULT * df['atr'],
                np.where(sign == -1, df['close'] - ATR_TP_MULT * df['atr'], np.nan))
    return df

6) Backtester

In [16]:
import pandas as pd

class Backtester:
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.equity = INITIAL_CAPITAL
        self.trades = []
        self.equity_curve = []

    def run(self):
        pos = None
        entry_price = sl = tp = None
        entry_time = None
        for ts, row in self.df.iterrows():
            self.equity_curve.append({'timestamp': ts, 'equity': self.equity})
            if pos is not None:
                hit_sl = (pos == 1 and row['low'] <= sl) or (pos == -1 and row['high'] >= sl)
                hit_tp = (pos == 1 and row['high'] >= tp) or (pos == -1 and row['low'] <= tp)
                if hit_sl or hit_tp:
                    exit_price = sl if hit_sl else tp
                    pct = (exit_price - entry_price) / entry_price * pos
                    rb = ATR_SL_MULT * row['atr'] / max(entry_price, 1e-9)
                    pnl = self.equity * RISK_PCT / 100 * pct / max(rb, 1e-9)
                    self.equity += pnl
                    self.trades.append({
                        'entry_time': entry_time,
                        'exit_time': ts,
                        'direction': 'LONG' if pos == 1 else 'SHORT',
                        'entry_price': entry_price,
                        'exit_price': exit_price,
                        'sl': sl, 'tp': tp, 'pnl': pnl,
                        'result': 'WIN' if hit_tp else 'LOSS'
                    })
                    pos = None
            if pos is None and row['signal'] != 0 and not (LONG_ONLY and row['signal'] == -1):
                pos = row['signal']
                entry_price = row['close']
                entry_time = ts
                sl = row['sl']
                tp = row['tp']
        return self._stats()

    def _stats(self):
        if not self.trades:
            return {'error': 'No trades — try lowering ADX_THRESHOLD or extending lookback'}
        df = pd.DataFrame(self.trades)
        wins = df[df['result'] == 'WIN']
        losses = df[df['result'] == 'LOSS']
        eq = pd.DataFrame(self.equity_curve).set_index('timestamp')
        dd = (eq['equity'] - eq['equity'].cummax()) / eq['equity'].cummax() * 100
        ret = df['pnl'] / INITIAL_CAPITAL
        sh = (ret.mean()/ret.std()*np.sqrt(252)) if ret.std() > 0 else 0
        return {
            'total_trades': int(len(df)),
            'win_rate': round(len(wins)/len(df)*100, 2) if len(df) else 0.0,
            'total_return': round((self.equity-INITIAL_CAPITAL)/INITIAL_CAPITAL*100, 2),
            'final_equity': round(self.equity, 2),
            'sharpe_ratio': round(sh, 2),
            'max_drawdown': round(float(dd.min()) if not dd.empty else 0.0, 2),
            'avg_win': round(float(wins['pnl'].mean()) if len(wins) else 0.0, 2),
            'avg_loss': round(float(losses['pnl'].mean()) if len(losses) else 0.0, 2),
            'profit_factor': round(float(wins['pnl'].sum()/abs(losses['pnl'].sum())) if len(losses) and abs(losses['pnl'].sum())>0 else float('inf'), 2),
            'trades_df': df,
            'equity_curve': eq,
        }

7) Email Notifier

In [17]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime

def send_trade_email(direction: str, entry_price: float, sl: float, tp: float,
                     size: float, risk_dollars: float, balance: float,
                     symbol: str, ts: str):
    """
    Sends an email alert when a live trade is opened.
    Silently skips if email is disabled or credentials are missing.

    Parameters
    ----------
    direction     : 'LONG' or 'SHORT'
    entry_price   : fill price
    sl            : stop-loss price
    tp            : take-profit price
    size          : position size in base asset (e.g. BTC)
    risk_dollars  : dollar amount risked on this trade
    balance       : current available balance BEFORE this trade
    symbol        : e.g. 'BTC/USDT'
    ts            : timestamp string
    """
    if not ENABLE_EMAIL:
        return
    if not all([EMAIL_SENDER, EMAIL_PASSWORD, EMAIL_RECIPIENT]):
        log.warning("Email skipped — credentials not fully configured.")
        return

    rr_ratio   = round(ATR_TP_MULT / ATR_SL_MULT, 2)
    sl_dist    = round(abs(entry_price - sl), 4)
    tp_dist    = round(abs(tp - entry_price), 4)
    emoji      = "📈" if direction == 'LONG' else "📉"

    subject = f"{emoji} [{direction}] {symbol} trade opened — {ts}"

    body = f"""<html><body style="font-family:Arial,sans-serif;color:#222;">
<h2 style="color:{'#16a34a' if direction=='LONG' else '#dc2626'}">
  {emoji} {direction} — {symbol}
</h2>
<table cellpadding="6" style="border-collapse:collapse;min-width:340px;">
  <tr style="background:#f3f4f6">
    <td><b>Time</b></td><td>{ts}</td>
  </tr>
  <tr>
    <td><b>Direction</b></td><td>{direction}</td>
  </tr>
  <tr style="background:#f3f4f6">
    <td><b>Entry price</b></td><td>{entry_price:,.4f}</td>
  </tr>
  <tr>
    <td><b>Stop-loss</b></td><td>{sl:,.4f} &nbsp;({sl_dist:,.4f} away)</td>
  </tr>
  <tr style="background:#f3f4f6">
    <td><b>Take-profit</b></td><td>{tp:,.4f} &nbsp;({tp_dist:,.4f} away)</td>
  </tr>
  <tr>
    <td><b>R:R ratio</b></td><td>1 : {rr_ratio}</td>
  </tr>
  <tr style="background:#f3f4f6">
    <td><b>Position size</b></td><td>{size:.6f} {symbol.split('/')[0]}</td>
  </tr>
  <tr>
    <td><b>Risk amount</b></td><td>${risk_dollars:,.2f} ({RISK_PCT}% of balance)</td>
  </tr>
  <tr style="background:#f3f4f6">
    <td><b>Balance before trade</b></td><td>${balance:,.2f} {LIVE_QUOTE_ASSET}</td>
  </tr>
  <tr>
    <td><b>Leverage</b></td><td>{LIVE_LEVERAGE}x</td>
  </tr>
  <tr style="background:#f3f4f6">
    <td><b>Timeframes</b></td><td>{LTF} / {HTF}{'/ ' + HTF2 if USE_HTF2 else ''}</td>
  </tr>
</table>
<p style="color:#6b7280;font-size:12px;margin-top:16px;">
  Sent automatically by your trading bot. This is not financial advice.
</p>
</body></html>"""

    try:
        msg = MIMEMultipart('alternative')
        msg['Subject'] = subject
        msg['From']    = EMAIL_SENDER
        msg['To']      = EMAIL_RECIPIENT
        msg.attach(MIMEText(body, 'html'))

        with smtplib.SMTP(EMAIL_SMTP_HOST, EMAIL_SMTP_PORT) as server:
            server.ehlo()
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.sendmail(EMAIL_SENDER, EMAIL_RECIPIENT, msg.as_string())

        log.info(f"Email sent → {EMAIL_RECIPIENT}")
    except Exception as e:
        log.error(f"Email failed: {e}")

8) Paper Trader

In [18]:
import time

class PaperTrader:
    def __init__(self):
        self.equity      = INITIAL_CAPITAL
        self.position    = None
        self.entry_price = None
        self.entry_time  = None
        self.sl          = None
        self.tp          = None
        self.trade_log   = []
        self.iteration   = 0

    def _unrealised_pct(self, price: float) -> float:
        if self.position == 'LONG':  return (price - self.entry_price) / self.entry_price * 100
        if self.position == 'SHORT': return (self.entry_price - price) / self.entry_price * 100
        return 0.0

    def _record_exit(self, ts, row, exit_price, hit_tp: bool):
        sign = 1 if self.position == 'LONG' else -1
        pct  = (exit_price - self.entry_price) / self.entry_price * sign
        rb   = ATR_SL_MULT * row['atr'] / max(self.entry_price, 1e-9)
        pnl  = self.equity * RISK_PCT / 100 * pct / max(rb, 1e-9)
        self.equity += pnl
        self.trade_log.append({
            'entry_time': self.entry_time, 'exit_time': ts,
            'direction': self.position,
            'entry_price': self.entry_price, 'exit_price': exit_price,
            'sl': self.sl, 'tp': self.tp, 'pnl': pnl,
            'result': 'WIN' if hit_tp else 'LOSS'
        })
        self.position = self.entry_price = self.entry_time = self.sl = self.tp = None

    def run(self):
        tf_s = {'1m':60,'5m':300,'15m':900,'30m':1800,'1h':3600,'4h':14400,'1d':86400}
        poll = PAPER_POLL_INTERVAL or tf_s.get(LTF, 3600)
        htf2_info = f" + {HTF2}" if USE_HTF2 else ""
        print(f"Paper trader started — {BROKER.upper()} {SYMBOL} {LTF}/{HTF}{htf2_info}\n"
              f"Virtual capital: ${INITIAL_CAPITAL:,.2f}  Poll: {poll}s  Ctrl+C to stop\n")
        while True:
            try:
                self.iteration += 1
                if DATA_PULL_MODE == 'bars':
                    ltf_raw  = get_data(SYMBOL, LTF,  bars=LTF_BARS)
                    htf_raw  = get_data(SYMBOL, HTF,  bars=HTF_BARS)
                    htf2_raw = get_data(SYMBOL, HTF2, bars=HTF2_BARS) if USE_HTF2 else None
                else:
                    ltf_raw  = get_data(SYMBOL, LTF,  days=PAPER_LTF_DAYS)
                    htf_raw  = get_data(SYMBOL, HTF,  days=PAPER_HTF_DAYS)
                    htf2_raw = get_data(SYMBOL, HTF2, days=PAPER_HTF2_DAYS) if USE_HTF2 else None
                ltf  = compute_indicators(ltf_raw)
                htf  = compute_indicators(htf_raw)
                htf2 = compute_indicators(htf2_raw) if USE_HTF2 and htf2_raw is not None else None
                row      = ltf.iloc[-1]
                prev     = ltf.iloc[-2]
                ts       = ltf.index[-1]
                htf_bull = htf.iloc[-1]['ema_fast'] > htf.iloc[-1]['ema_slow']
                htf2_bull = (htf2.iloc[-1]['ema_fast'] > htf2.iloc[-1]['ema_slow']) if htf2 is not None else None
                cross_up   = (row['ema_fast'] > row['ema_slow']) and (prev['ema_fast'] <= prev['ema_slow'])
                cross_down = (row['ema_fast'] < row['ema_slow']) and (prev['ema_fast'] >= prev['ema_slow'])
                adx_ok = row['adx'] > ADX_THRESHOLD
                atr_ok = row['atr'] > row['atr_ma']
                if USE_HTF2 and htf2_bull is not None:
                    long_ok  = cross_up   and adx_ok and atr_ok and htf_bull        and htf2_bull
                    short_ok = cross_down and adx_ok and atr_ok and (not htf_bull)  and (not htf2_bull) and not LONG_ONLY
                else:
                    long_ok  = cross_up   and adx_ok and atr_ok and htf_bull
                    short_ok = cross_down and adx_ok and atr_ok and (not htf_bull) and not LONG_ONLY
                if self.position:
                    hit_sl = (self.position=='LONG'  and row['low']  <= self.sl) or \
                             (self.position=='SHORT' and row['high'] >= self.sl)
                    hit_tp = (self.position=='LONG'  and row['high'] >= self.tp) or \
                             (self.position=='SHORT' and row['low']  <= self.tp)
                    if hit_sl or hit_tp:
                        self._record_exit(ts, row, self.sl if hit_sl else self.tp, hit_tp)
                        self._print_status(ts, row, htf_bull, htf2_bull, cross_up, cross_down)
                        if PAPER_MAX_TRADES and len(self.trade_log) >= PAPER_MAX_TRADES:
                            print(f"Reached max trades ({PAPER_MAX_TRADES}). Stopping.")
                            break
                        time.sleep(poll); continue
                    self._print_status(ts, row, htf_bull, htf2_bull, cross_up, cross_down)
                    time.sleep(poll); continue
                if long_ok or short_ok:
                    self.position    = 'LONG' if long_ok else 'SHORT'
                    self.entry_price = row['close']
                    self.entry_time  = ts
                    sgn = 1 if long_ok else -1
                    self.sl = row['close'] - sgn * ATR_SL_MULT * row['atr']
                    self.tp = row['close'] + sgn * ATR_TP_MULT * row['atr']
                self._print_status(ts, row, htf_bull, htf2_bull, cross_up, cross_down)
            except KeyboardInterrupt:
                print("\nStopped by user.")
                break
            except Exception as e:
                log.error(f"Loop error: {e}")
            time.sleep(poll)

    def _print_status(self, ts, row, htf_bull, htf2_bull, cross_up, cross_down):
        total_pnl = self.equity - INITIAL_CAPITAL
        win_count = sum(1 for t in self.trade_log if t['result']=='WIN')
        n_trades  = len(self.trade_log)
        win_rate  = (win_count/n_trades*100) if n_trades else 0.0
        unreal    = self._unrealised_pct(row['close']) if self.position else 0.0
        htf_label  = f"HTF({HTF}): {'BULL' if htf_bull else 'BEAR'}"
        htf2_label = f"  HTF2({HTF2}): {'BULL' if htf2_bull else 'BEAR'}" if USE_HTF2 and htf2_bull is not None else ""
        cross_str  = 'UP' if cross_up else ('DOWN' if cross_down else 'NONE')
        print("-"*68)
        print(f"Candle #{self.iteration} @ {ts.strftime('%Y-%m-%d %H:%M:%S')}  Price {row['close']:.5f}")
        print(f"EMA{EMA_FAST} {row['ema_fast']:.5f}  EMA{EMA_SLOW} {row['ema_slow']:.5f}  ADX {row['adx']:.2f}  ATR {row['atr']:.5f}")
        print(f"{htf_label}{htf2_label}  Cross: {cross_str}")
        if self.position:
            print(f"Open {self.position}  Entry {self.entry_price:.5f}  SL {self.sl:.5f}  TP {self.tp:.5f}  Unrl {unreal:+.2f}%")
        else:
            print("Open position: None")
        print(f"Equity ${self.equity:,.2f}  TotalPnL {total_pnl:+.2f}  Trades {n_trades}  WinRate {win_rate:.1f}%")

9) Live Trader

Real order execution via CCXT. Dynamic sizing: fetches live balance from the
exchange before every trade and calculates position size from it.

**Position sizing formula (per trade):**
```
balance      = exchange.fetch_balance()[LIVE_QUOTE_ASSET]['free']
risk_dollars = balance * RISK_PCT / 100
sl_distance  = abs(entry_price - sl_price)          # in price units
sl_pct       = sl_distance / entry_price             # as a fraction
size         = (risk_dollars / sl_pct) / entry_price * LIVE_LEVERAGE
```
This sizes the trade so that if SL is hit, you lose exactly RISK_PCT% of your
current balance — not a fixed dollar amount, but a fixed % of whatever you have
at the moment the trade opens.

In [19]:
import time

class LiveTrader:
    """
    Real order execution on Binance or Kraken via CCXT.

    Order flow per signal:
      1. Fetch live balance from exchange
      2. Calculate position size from RISK_PCT of that balance
      3. Submit market entry order
      4. Submit stop-loss limit order (reduce-only / close)
      5. Submit take-profit limit order (reduce-only / close)
      6. Send email notification

    On exit (SL or TP hit on exchange):
      - Bot detects no open position on next poll
      - Cancels any remaining open SL/TP orders
      - Resets state and waits for next signal
    """

    def __init__(self):
        import ccxt
        # Build authenticated exchange client
        if BROKER == 'binance':
            self.ex = ccxt.binance({
                'apiKey': BINANCE_API_KEY,
                'secret': BINANCE_API_SECRET,
                'enableRateLimit': True,
                'options': {'defaultType': 'future'},
            })
        elif BROKER == 'kraken':
            self.ex = ccxt.kraken({
                'apiKey': KRAKEN_API_KEY,
                'secret': KRAKEN_API_SECRET,
                'enableRateLimit': True,
            })
        else:
            raise ValueError(f"Live mode not supported for BROKER='{BROKER}'. Use binance or kraken.")

        self.ex.load_markets()
        # ── Set leverage on exchange to match config ──────────────────────────────
        # This runs once at startup and ensures the exchange leverage always
        # matches LIVE_LEVERAGE in your config — prevents mismatch after restarts.
        
        if BROKER in ('binance', 'bybit'):
            try:
                self.ex.set_leverage(int(LIVE_LEVERAGE), SYMBOL)
                log.info(f"Leverage set to {int(LIVE_LEVERAGE)}x on {SYMBOL}")
            except Exception as e:
                log.warning(f"Could not set leverage automatically: {e} — set it manually on the exchange.")
        self.position      = None   # 'LONG' | 'SHORT' | None
        self.entry_price   = None
        self.sl            = None
        self.tp            = None
        self.entry_time    = None
        self.order_size    = None   # base asset size of open position
        self.sl_order_id   = None
        self.tp_order_id   = None
        self.trade_log     = []
        self.iteration     = 0
        log.info(f"LiveTrader ready — {BROKER.upper()} {SYMBOL}")

        # ── Startup position check ────────────────────────────────────────
        # Runs once on init. If a position is already open on the exchange
        # (e.g. after a crash or kernel restart), restore local state so the
        # bot never opens a second trade on top of an existing one.
        self._restore_position_on_startup()

    # ── Startup restore ───────────────────────────────────────────────────────

    def _restore_position_on_startup(self):
        """
        Called once at startup. Queries the exchange for any open position
        on SYMBOL and restores local state if one is found.

        This prevents the bot from opening a second trade after a crash,
        kernel restart, or accidental notebook re-run.

        What it restores:
          - self.position    : 'LONG' or 'SHORT'
          - self.entry_price : average entry price from the position
          - self.order_size  : current position size in base asset
          - self.sl / tp     : reconstructed from ATR_SL_MULT / ATR_TP_MULT
                               (best estimate — exact original levels not
                                stored on exchange)

        What it also does:
          - Re-links any existing open SL/TP orders by scanning open orders
            and matching them to the restored position direction.
        """
        log.info("Startup check: querying exchange for open positions...")
        try:
            # ── 1. Check for open futures position ───────────────────────────
            positions = self.ex.fetch_positions([SYMBOL])
            open_pos  = None
            for p in positions:
                size = float(p.get('contracts', 0) or p.get('contractSize', 0) or 0)
                if size != 0:
                    open_pos = p
                    break

            if open_pos is None:
                log.info("Startup check: no open position found. Starting fresh.")
                return

            # ── 2. Determine direction ────────────────────────────────────────
            side         = open_pos.get('side', '').lower()   # 'long' or 'short'
            direction    = 'LONG' if side == 'long' else 'SHORT'
            entry_price  = float(open_pos.get('entryPrice') or open_pos.get('averagePrice') or 0)
            size         = abs(float(open_pos.get('contracts', 0) or open_pos.get('contractSize', 0)))

            if entry_price == 0 or size == 0:
                log.warning("Startup check: found position but could not read entry price or size. Starting fresh.")
                return

            # ── 3. Reconstruct SL / TP from entry price ───────────────────────
            # We don't know the original ATR value, so we reconstruct SL/TP
            # using the current ATR multipliers and a fixed % of entry price
            # as a proxy. The bot will re-link real orders below if they exist.
            sgn      = 1 if direction == 'LONG' else -1
            # Use ATR_SL_MULT * 0.02 * entry as a rough SL distance proxy
            # (0.02 = ~2% move, scaled by your SL multiplier)
            sl_proxy = entry_price - sgn * (ATR_SL_MULT / ATR_TP_MULT) * entry_price * 0.02
            tp_proxy = entry_price + sgn * entry_price * 0.02

            # ── 4. Re-link open SL / TP orders ───────────────────────────────
            sl_order_id = None
            tp_order_id = None
            try:
                open_orders = self.ex.fetch_open_orders(SYMBOL)
                for o in open_orders:
                    o_side  = o.get('side', '').lower()
                    o_type  = o.get('type', '').lower()
                    close_side = 'sell' if direction == 'LONG' else 'buy'
                    if o_side == close_side:
                        if 'stop' in o_type or 'trigger' in o_type:
                            sl_order_id = o['id']
                            sl_proxy    = float(o.get('stopPrice') or o.get('triggerPrice') or sl_proxy)
                            log.info(f"Startup check: re-linked SL order {sl_order_id} @ {sl_proxy:.4f}")
                        elif 'limit' in o_type:
                            tp_order_id = o['id']
                            tp_proxy    = float(o.get('price') or tp_proxy)
                            log.info(f"Startup check: re-linked TP order {tp_order_id} @ {tp_proxy:.4f}")
            except Exception as e:
                log.warning(f"Startup check: could not fetch open orders: {e}")

            # ── 5. Restore local state ────────────────────────────────────────
            self.position      = direction
            self.entry_price   = entry_price
            self.order_size    = size
            self.sl            = sl_proxy
            self.tp            = tp_proxy
            self.sl_order_id   = sl_order_id
            self.tp_order_id   = tp_order_id
            self.entry_time    = 'restored-on-startup'

            log.warning(
                f"Startup check: RESTORED existing {direction} position — "
                f"entry={entry_price:.4f}  size={size}  "
                f"SL={sl_proxy:.4f}  TP={tp_proxy:.4f}  "
                f"Bot will monitor this position and NOT open a new one."
            )

        except Exception as e:
            # Never block startup on a failed check — just log and continue
            log.error(f"Startup position check failed: {e} — assuming no open position.")

    # ── Balance & sizing ──────────────────────────────────────────────────────

    def fetch_balance(self) -> float:
        """Return free balance of LIVE_QUOTE_ASSET (e.g. USDT) from the exchange."""
        bal = self.ex.fetch_balance()
        free = bal.get(LIVE_QUOTE_ASSET, {}).get('free', 0.0)
        return float(free)

    def calc_position_size(self, entry_price: float, sl_price: float, balance: float) -> float:
        """
        Size the position so that a stop-loss hit costs exactly RISK_PCT% of balance.

        risk_dollars = balance * RISK_PCT / 100
        sl_distance  = |entry - sl| in price units
        sl_pct       = sl_distance / entry_price
        size (base)  = (risk_dollars / sl_pct) / entry_price * leverage

        Example: balance=$500, RISK_PCT=5%, entry=95000, sl=93290 (1.8 ATR)
          risk_dollars = $25
          sl_pct       = 1800/95000 = 0.01895
          notional     = 25 / 0.01895 = $1319
          size         = 1319 / 95000 = 0.01388 BTC
        """
        risk_dollars = balance * RISK_PCT / 100.0
        sl_distance  = abs(entry_price - sl_price)
        if sl_distance <= 0:
            raise ValueError("SL distance is zero — cannot size position.")
        sl_pct   = sl_distance / entry_price
        notional = risk_dollars / sl_pct
        size     = notional / entry_price * LIVE_LEVERAGE
        # Round to exchange precision
        market   = self.ex.market(SYMBOL)
        size     = self.ex.amount_to_precision(SYMBOL, size)
        return float(size)

    # ── Order helpers ─────────────────────────────────────────────────────────

    def _place_entry(self, direction: str, size: float) -> dict:
        side = 'buy' if direction == 'LONG' else 'sell'
        order = self.ex.create_market_order(SYMBOL, side, size)
        log.info(f"Entry {side.upper()} {size} {SYMBOL} — order id: {order['id']}")
        return order

    def _place_sl(self, direction: str, size: float, sl_price: float):
        """Place a stop-loss order. Side is opposite to position direction."""
        side   = 'sell' if direction == 'LONG' else 'buy'
        params = {'stopPrice': sl_price, 'type': 'stop_market'}
        try:
            order = self.ex.create_order(SYMBOL, 'stop_market', side, size,
                                          price=sl_price, params=params)
            log.info(f"SL order placed @ {sl_price:.4f} — id: {order['id']}")
            return order['id']
        except Exception as e:
            log.warning(f"SL order failed (may need futures account): {e}")
            return None

    def _place_tp(self, direction: str, size: float, tp_price: float):
        """Place a take-profit limit order. Side is opposite to position direction."""
        side = 'sell' if direction == 'LONG' else 'buy'
        try:
            order = self.ex.create_limit_order(SYMBOL, side, size, tp_price)
            log.info(f"TP order placed @ {tp_price:.4f} — id: {order['id']}")
            return order['id']
        except Exception as e:
            log.warning(f"TP order failed: {e}")
            return None

    def _cancel_open_orders(self):
        """Cancel any remaining SL / TP orders when position closes."""
        for oid in [self.sl_order_id, self.tp_order_id]:
            if oid:
                try:
                    self.ex.cancel_order(oid, SYMBOL)
                    log.info(f"Cancelled order {oid}")
                except Exception as e:
                    log.warning(f"Cancel order {oid} failed (may already be filled): {e}")
        self.sl_order_id = self.tp_order_id = None

    def _check_position_still_open(self) -> bool:
        """
        Returns True if we still hold the base asset (for LONG)
        or if the short position is still open (for futures).
        For spot, checks if base asset balance > dust threshold.
        """
        if self.position is None:
            return False
        try:
            bal = self.ex.fetch_balance()
            base_asset = SYMBOL.split('/')[0]   # e.g. 'BTC'
            if self.position == 'LONG':
                held = float(bal.get(base_asset, {}).get('free', 0.0))
                return held >= self.order_size * 0.95   # 5% tolerance for fees
            else:
                # SHORT on spot is not standard — use open orders as proxy
                open_orders = self.ex.fetch_open_orders(SYMBOL)
                return len(open_orders) > 0
        except Exception as e:
            log.warning(f"Position check failed: {e}")
            return True   # assume still open on error to be safe

    # ── Main loop ─────────────────────────────────────────────────────────────

    def run(self):
        poll = PAPER_POLL_INTERVAL
        htf2_info = f" + {HTF2}" if USE_HTF2 else ""
        balance   = self.fetch_balance()
        print(
            f"\nLive trader — {BROKER.upper()} {SYMBOL}  {LTF}/{HTF}{htf2_info}\n"
            f"Balance: ${balance:,.2f} {LIVE_QUOTE_ASSET}  Risk/Trade: {RISK_PCT}%  "
            f"Leverage: {LIVE_LEVERAGE}x  Poll: {poll}s\n"
            f"Ctrl+C to stop.\n"
        )

        while True:
            try:
                self.iteration += 1

                # ── Fetch market data ─────────────────────────────────────────
                if DATA_PULL_MODE == 'bars':
                    ltf_raw  = get_data(SYMBOL, LTF,  bars=LTF_BARS)
                    htf_raw  = get_data(SYMBOL, HTF,  bars=HTF_BARS)
                    htf2_raw = get_data(SYMBOL, HTF2, bars=HTF2_BARS) if USE_HTF2 else None
                else:
                    ltf_raw  = get_data(SYMBOL, LTF,  days=PAPER_LTF_DAYS)
                    htf_raw  = get_data(SYMBOL, HTF,  days=PAPER_HTF_DAYS)
                    htf2_raw = get_data(SYMBOL, HTF2, days=PAPER_HTF2_DAYS) if USE_HTF2 else None

                ltf  = compute_indicators(ltf_raw)
                htf  = compute_indicators(htf_raw)
                htf2 = compute_indicators(htf2_raw) if USE_HTF2 and htf2_raw is not None else None

                row, prev = ltf.iloc[-1], ltf.iloc[-2]
                ts        = ltf.index[-1]
                htf_bull  = htf.iloc[-1]['ema_fast'] > htf.iloc[-1]['ema_slow']
                htf2_bull = (htf2.iloc[-1]['ema_fast'] > htf2.iloc[-1]['ema_slow']) if htf2 is not None else None

                cross_up   = (row['ema_fast'] > row['ema_slow']) and (prev['ema_fast'] <= prev['ema_slow'])
                cross_down = (row['ema_fast'] < row['ema_slow']) and (prev['ema_fast'] >= prev['ema_slow'])
                adx_ok     = row['adx'] > ADX_THRESHOLD
                atr_ok     = row['atr'] > row['atr_ma']

                if USE_HTF2 and htf2_bull is not None:
                    long_ok  = cross_up   and adx_ok and atr_ok and htf_bull        and htf2_bull
                    short_ok = cross_down and adx_ok and atr_ok and (not htf_bull)  and (not htf2_bull) and not LONG_ONLY
                else:
                    long_ok  = cross_up   and adx_ok and atr_ok and htf_bull
                    short_ok = cross_down and adx_ok and atr_ok and (not htf_bull) and not LONG_ONLY

                # ── Check if existing position is still open ──────────────────
                if self.position:
                    still_open = self._check_position_still_open()
                    if not still_open:
                        log.info(f"{self.position} position closed on exchange (SL or TP hit).")
                        self._cancel_open_orders()
                        self.trade_log.append({
                            'direction'  : self.position,
                            'entry_price': self.entry_price,
                            'entry_time' : self.entry_time,
                            'closed_at'  : str(ts),
                            'sl': self.sl, 'tp': self.tp,
                        })
                        self.position = self.entry_price = self.entry_time = None
                        self.sl = self.tp = self.order_size = None
                    self._print_status(ts, row, htf_bull, htf2_bull, cross_up, cross_down)
                    time.sleep(poll)
                    continue

                # ── Open new position if signal fires ─────────────────────────
                if long_ok or short_ok:
                    direction    = 'LONG' if long_ok else 'SHORT'
                    entry_price  = row['close']
                    sgn          = 1 if long_ok else -1
                    sl_price     = entry_price - sgn * ATR_SL_MULT * row['atr']
                    tp_price     = entry_price + sgn * ATR_TP_MULT * row['atr']

                    # ── Dynamic sizing: fetch live balance now ────────────────
                    balance      = self.fetch_balance()
                    risk_dollars = balance * RISK_PCT / 100.0
                    size         = self.calc_position_size(entry_price, sl_price, balance)

                    log.info(
                        f"Signal: {direction}  entry={entry_price:.4f}  "
                        f"sl={sl_price:.4f}  tp={tp_price:.4f}  "
                        f"size={size}  risk=${risk_dollars:.2f}  balance=${balance:.2f}"
                    )

                    # ── Place orders ──────────────────────────────────────────
                    entry_order       = self._place_entry(direction, size)
                    filled_price      = float(entry_order.get('average') or entry_order.get('price') or entry_price)
                    self.sl_order_id  = self._place_sl(direction, size, sl_price)
                    self.tp_order_id  = self._place_tp(direction, size, tp_price)

                    # ── Update local state ────────────────────────────────────
                    self.position    = direction
                    self.entry_price = filled_price
                    self.entry_time  = ts
                    self.sl          = sl_price
                    self.tp          = tp_price
                    self.order_size  = float(size)

                    # ── Send email notification ───────────────────────────────
                    send_trade_email(
                        direction    = direction,
                        entry_price  = filled_price,
                        sl           = sl_price,
                        tp           = tp_price,
                        size         = float(size),
                        risk_dollars = risk_dollars,
                        balance      = balance,
                        symbol       = SYMBOL,
                        ts           = str(ts),
                    )

                self._print_status(ts, row, htf_bull, htf2_bull, cross_up, cross_down)

            except KeyboardInterrupt:
                print("\nStopped by user.")
                print("Note: any open SL/TP orders on the exchange remain active.")
                break
            except Exception as e:
                log.error(f"Loop error: {e}")
            time.sleep(poll)

    def _print_status(self, ts, row, htf_bull, htf2_bull, cross_up, cross_down):
        n_trades   = len(self.trade_log)
        htf_label  = f"HTF({HTF}): {'BULL' if htf_bull else 'BEAR'}"
        htf2_label = f"  HTF2({HTF2}): {'BULL' if htf2_bull else 'BEAR'}" if USE_HTF2 and htf2_bull is not None else ""
        cross_str  = 'UP' if cross_up else ('DOWN' if cross_down else 'NONE')
        try:
            balance = self.fetch_balance()
            bal_str = f"${balance:,.2f} {LIVE_QUOTE_ASSET}"
        except Exception:
            bal_str = "(fetch failed)"
        print("-"*70)
        print(f"[LIVE] #{self.iteration} @ {ts.strftime('%Y-%m-%d %H:%M:%S')}  Price {row['close']:.4f}")
        print(f"EMA{EMA_FAST} {row['ema_fast']:.4f}  EMA{EMA_SLOW} {row['ema_slow']:.4f}  ADX {row['adx']:.2f}  ATR {row['atr']:.4f}")
        print(f"{htf_label}{htf2_label}  Cross: {cross_str}")
        if self.position:
            print(f"Position: {self.position}  Entry {self.entry_price:.4f}  SL {self.sl:.4f}  TP {self.tp:.4f}  Size {self.order_size}")
        else:
            print("Position: None")
        print(f"Exchange balance: {bal_str}  Completed trades: {n_trades}")

10) Runner (Backtest / Paper / Live)

In [20]:
from math import sqrt

def run_backtest():
    htf2_info = f" + {HTF2}" if USE_HTF2 else ""
    print(f"Fetching {SYMBOL}  LTF:{LTF}  HTF:{HTF}{htf2_info}")
    if DATA_PULL_MODE == 'bars':
        ltf_raw  = get_data(SYMBOL, LTF,  bars=LTF_BARS)
        htf_raw  = get_data(SYMBOL, HTF,  bars=HTF_BARS)
        htf2_raw = get_data(SYMBOL, HTF2, bars=HTF2_BARS) if USE_HTF2 else None
    else:
        ltf_raw  = get_data(SYMBOL, LTF,  days=BACKTEST_DAYS)
        htf_raw  = get_data(SYMBOL, HTF,  days=max(BACKTEST_DAYS*2, BACKTEST_DAYS+7))
        htf2_raw = get_data(SYMBOL, HTF2, days=max(BACKTEST_DAYS*4, BACKTEST_DAYS+30)) if USE_HTF2 else None
    ltf  = compute_indicators(ltf_raw)
    htf  = compute_indicators(htf_raw)
    htf2 = compute_indicators(htf2_raw) if USE_HTF2 and htf2_raw is not None else None
    df   = generate_signals(ltf, htf, htf2)
    bt   = Backtester(df)
    stats = bt.run()
    if 'error' in stats:
        print('WARNING:', stats['error'])
        return
    tf_label    = f"{LTF} -> {HTF}" + (f" -> {HTF2}" if USE_HTF2 else "")
    htf2_status = f"ON ({HTF2})" if USE_HTF2 else "OFF"
    print("\n" + "="*40)
    print("BACKTEST RESULTS")
    print("="*40)
    print(f"Symbol         : {SYMBOL}")
    print(f"Timeframes     : {tf_label}")
    print(f"HTF2 filter    : {htf2_status}")
    long_only_status = "ON (shorts skipped)" if LONG_ONLY else "OFF (both directions)"
    print(f"Long-only mode : {long_only_status}")
    print(f"Total trades   : {stats['total_trades']}")
    print(f"Win rate       : {stats['win_rate']}%")
    print(f"Total return   : {stats['total_return']}%")
    print(f"Innitial Capital : {INITIAL_CAPITAL:,.2f}")
    print(f"Final equity   : ${stats['final_equity']}")
    print(f"Sharpe ratio   : {stats['sharpe_ratio']}")
    print(f"Max drawdown   : {stats['max_drawdown']}%")
    print(f"Profit factor  : {stats['profit_factor']}")
    print(f"Avg win        : ${stats['avg_win']}")
    print(f"Avg loss       : ${stats['avg_loss']}")
    print(f"Take Profit    : {ATR_TP_MULT}")
    print(f"Stop Loss      : {ATR_SL_MULT}")
    print(f"Backtest days  : {BACKTEST_DAYS}")
    print("="*40 + "\n")
    
    print(stats['trades_df'].to_string(index=False))

if MODE == 'backtest':
    run_backtest()
elif MODE == 'paper':
    PaperTrader().run()
elif MODE == 'live':
    LiveTrader().run()
else:
    print(f"Unknown MODE='{MODE}'. Options: backtest | paper | live")
    

Fetching BTC/USDT:USDT  LTF:1m  HTF:30m
Data pull BTC/USDT:USDT 1m   src: BINANCE  candles:  43200  span: 29d (2026-02-14 -> 2026-03-16)  [execute on: BINANCE]
Data pull BTC/USDT:USDT 30m  src: BINANCE  candles:   2880  span: 59d (2026-01-15 -> 2026-03-16)  [execute on: BINANCE]

BACKTEST RESULTS
Symbol         : BTC/USDT:USDT
Timeframes     : 1m -> 30m
HTF2 filter    : OFF
Long-only mode : OFF (both directions)
Total trades   : 58
Win rate       : 62.07%
Total return   : 63.95%
Innitial Capital : 100.00
Final equity   : $163.95
Sharpe ratio   : 3.8
Max drawdown   : -16.05%
Profit factor  : 1.64
Avg win        : $4.53
Avg loss       : $-4.51
Take Profit    : 1.7
Stop Loss      : 1.5
Backtest days  : 30

         entry_time           exit_time direction  entry_price   exit_price           sl           tp       pnl result
2026-02-16 00:24:00 2026-02-16 00:32:00     SHORT      68752.8 68839.678334 68839.678334 68654.337888 -3.461235   LOSS
2026-02-16 05:03:00 2026-02-16 05:06:00     SHORT